# Partie 3 - Génération de données synthétiques

## *Canevas du rapport à rendre pour le 06/04/2026*

**Génération de données synthétiques par échantillonage dans l'espace latent**

* Objectif : Créer des données synthétiques qui conservent l'information des données patients tout
en protégeant la confidentialité de leurs données individuelles
* Principe
  - principe de l'ACP
  - principe de l'échantillonage dans l'espace de l'ACP
  - approche de Guillaudeux *et al.* (2023) (réexpliquer avec vos mots)
* Application
  - code
  - évaluation de vos résultats
      - Montrer que vos données resemblent aux données initiales (code, graphiques, tables)
      - Montrer que les corrélations entre les variables sont conservées (idem)
      - Montrer que les données patients (originales) ne se retrouvent pas dans les données synthétiques (idem)
* Conclusion

Référence : [Guillaudeux *et al.* (2023)](https://www.nature.com/articles/s41746-023-00771-5)

*By mathematically simulating the whole individual observation,
synthetic datasets protect individual privacy while attempting to
retain the statistical relevance of the dataset. Synthetic data are
defined as any production data not obtained from real measurements.
In practice, these data are drawn at random using data
models whose objective is to mimic a real dataset or an individual
observation.*

In [1]:
# bibliothèque de calcul scientifique
import numpy as np

# bibliothèque pour l'analyse des données
import pandas as pd
import sklearn as skl

# bibliothèques pour l'exploration des données
import ipywidgets as widgets
import plotly.graph_objs as go
import plotly.express as px

In [2]:
print(f'NumPy version {np.__version__}')
print(f'Pandas version {pd.__version__}')
print(f'scikit-learn version {skl.__version__}')

NumPy version 2.4.3
Pandas version 3.0.1
scikit-learn version 1.8.0


## I. Analyse en composantes principales (ACP)

L'analyse en composantes principales (en anglais, PCA pour *principal component analysis*) consiste à transformer les variables (*features*) réelles --- qui sont typiquement corrélées --- en variables latentes qui soient orthogonales entre elles, donc décorrélées les unes des autres. En plus de cette propriété géométrique (orthogonalité), c'est selon des propriétés statistiques que les dimensions (axes) portant ces variables latentes sont construites : les quelques premières dimensions contiennent l'essentiel de l'information (par exemple, 95% ou 99% de la variance des données). Ces `n_components` premières dimensions sont appelées « composantes principales » : il s'agit alors d'une méthode de [réduction de la dimensionnalité](https://fr.wikipedia.org/wiki/R%C3%A9duction_de_la_dimensionnalit%C3%A9).

### 1. Importer les données

In [3]:
iris = px.data.iris()

In [4]:
iris.sample(n=5)

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
94,5.6,2.7,4.2,1.3,versicolor,2
139,6.9,3.1,5.4,2.1,virginica,3
74,6.4,2.9,4.3,1.3,versicolor,2
93,5.0,2.3,3.3,1.0,versicolor,2
141,6.9,3.1,5.1,2.3,virginica,3


Utilisons un tableau de bord (*dashboard*) interactif afin d'explorer les données.

In [5]:
fig = go.FigureWidget()
keys = list(iris.keys()[:4])

@widgets.interact(x=keys, y=keys[::-1])
def update_px(x, y):
    p = px.scatter(iris, x, y, color='species', width=800, height=600)
    for i in range(len(p.data)):
        fig.data = []
    fig.update(data = [d.to_plotly_json() for d in p.data])
    fig.plotly_relayout(p.layout.to_plotly_json())
    
fig

interactive(children=(Dropdown(description='x', options=('sepal_length', 'sepal_width', 'petal_length', 'petal…

FigureWidget({
    'data': [{'hovertemplate': 'species=setosa<br>sepal_length=%{x}<br>petal_width=%{y}<extra></extra>',
              'legendgroup': 'setosa',
              'marker': {'color': '#636efa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'setosa',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'uid': '20729bc2-6dfb-4495-8bf6-323380cef6d5',
              'x': {'bdata': ('ZmZmZmZmFECamZmZmZkTQM3MzMzMzB' ... 'ZmZhJAMzMzMzMzFUAAAAAAAAAUQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('mpmZmZmZyT+amZmZmZnJP5qZmZmZmc' ... 'mZmck/mpmZmZmZyT+amZmZmZnJPw=='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'species=versicolor<br>sepal_length=%{x}<br>petal_width=%{y}<extra></extra>',
              'legendgroup': 'versicolor',
              'marker': {'color': '#EF553B', 'symbol': 'circle'

Si nous préférons visualiser toutes les relations deux à deux en même temps, nous devons
générer 16 graphiques (ou plutôt 6, en considérant les symétries et les relations triviales).

In [6]:
fig2 = go.FigureWidget()
cmaps = {'Plotly': px.colors.qualitative.Plotly, 'D3': px.colors.qualitative.D3,
         'Pastel': px.colors.qualitative.Pastel, 'Vivid': px.colors.qualitative.Vivid}

@widgets.interact(color_discrete_sequence=cmaps)
def update_cmap(color_discrete_sequence):
    p = px.scatter_matrix(
        iris,
        dimensions=keys,
        color='species',
        color_discrete_sequence=color_discrete_sequence,
        width=800,
        height=600,
    )
    for i in range(len(p.data)):
        fig2.data = []
    fig2.update(data = [d.to_plotly_json() for d in p.data])
    fig2.plotly_relayout(p.layout.to_plotly_json())
    
fig2

interactive(children=(Dropdown(description='color_discrete_sequence', options={'Plotly': ['#636EFA', '#EF553B'…

FigureWidget({
    'data': [{'dimensions': [{'axis': {'matches': True},
                              'label': 'sepal_length',
                              'values': {'bdata': ('ZmZmZmZmFECamZmZmZkTQM3MzMzMzB' ... 'ZmZhJAMzMzMzMzFUAAAAAAAAAUQA=='),
                                         'dtype': 'f8'}},
                             {'axis': {'matches': True},
                              'label': 'sepal_width',
                              'values': {'bdata': ('AAAAAAAADEAAAAAAAAAIQJqZmZmZmQ' ... 'mZmQlAmpmZmZmZDUBmZmZmZmYKQA=='),
                                         'dtype': 'f8'}},
                             {'axis': {'matches': True},
                              'label': 'petal_length',
                              'values': {'bdata': ('ZmZmZmZm9j9mZmZmZmb2P83MzMzMzP' ... 'ZmZvY/AAAAAAAA+D9mZmZmZmb2Pw=='),
                                         'dtype': 'f8'}},
                             {'axis': {'matches': True},
                              'label': 'petal_widt

### 2. Partitionner les données

In [7]:
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
0,5.1,3.5,1.4,0.2,setosa,1
1,4.9,3.0,1.4,0.2,setosa,1
2,4.7,3.2,1.3,0.2,setosa,1
3,4.6,3.1,1.5,0.2,setosa,1
4,5.0,3.6,1.4,0.2,setosa,1


In [8]:
X = iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]  # features

In [9]:
y = iris['species_id']  # target

In [10]:
X_train, X_test, y_train, y_test = skl.model_selection.train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

### 3. Préparer les données (*preprocessing*)

In [11]:
X_train.mean()

sepal_length    5.841667
sepal_width     3.044167
petal_length    3.770833
petal_width     1.204167
dtype: float64

In [12]:
X_train.std()

sepal_length    0.840926
sepal_width     0.445669
petal_length    1.767417
petal_width     0.763825
dtype: float64

Nous pourrions centrer et normaliser les *features* manuellement, en soustrayant la moyenne et en divisant par l'écart-type, mais nous connaissons désormais la classe `StandardScaler()` du module `preprocessing` de scikit-learn.

In [13]:
scaler = skl.preprocessing.StandardScaler()

In [14]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

### 4. et 5. Choisir, entraîner et appliquer un modèle de réduction de la dimensionnalité

La partition des données que nous avons effectuée serait utile dans un contexte de classification.
Dans l'idée :

In [ ]:
# clf = skl.given_model.GivenClassifier()
# model = clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)
# skl.metrics.accuracy_score(y_test, y_pred)

Peut-être aussi en vue de l'échantillonage (*training data vs. hold-out data*) ?

Pour l'instant, nous nous intéressons à la décomposition de la matrice $X$ (*features*)
dans l'espace de l'ACP, en considérant l'entièreté du jeu de données.

In [15]:
pca = skl.decomposition.PCA()

In [16]:
X = scaler.fit_transform(X)

In [17]:
pca.fit(X)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",None
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized S

In [18]:
pca.explained_variance_ratio_

array([0.72770452, 0.23030523, 0.03683832, 0.00515193])

In [19]:
pca.singular_values_

array([20.89551896, 11.75513248,  4.7013819 ,  1.75816839])

In [20]:
ev = pd.DataFrame(
    data={'component': np.arange(4),
          'percentage': pca.explained_variance_ratio_ * 100,
          'cumulative': np.cumsum(pca.explained_variance_ratio_) * 100}
)

In [21]:
ev

,component,percentage,cumulative
0,0,72.770452,72.770452
1,1,23.030523,95.800975
2,2,3.683832,99.484807
3,3,0.515193,100.000000


In [26]:
fig = px.bar(
    ev,
    x='component',
    y='percentage',
    labels={
        'component': 'Composante',
        'percentage': 'Pourcentage',
    },
    title='Proportion (%) de variance expliquée',
)
fig.update_layout(yaxis_range=[0, 100])
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [27]:
fig = px.line(
    ev,
    x='component',
    y='cumulative',
    markers=True,
    labels={
        'component': 'Composante',
        'cumulative': 'Pourcentage cumulé',
    },
    title='Variance expliquée cumulée (%)',
)
fig.update_layout(yaxis_range=[0, 110])
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [28]:
iris['species_id'].unique()

array([1, 2, 3])

In [29]:
iris['species'].unique()

<ArrowStringArray>
['setosa', 'versicolor', 'virginica']
Length: 3, dtype: str

In [30]:
X_latent = pca.fit_transform(X)

#### Exercice

Projetez les données "iris" sur les deux premières dimensions de l'espace latent
(c'est-à-dire l'espace de l'ACP).

### 6. Reconstruction des données

In [31]:
pca.components_  # coefficients des composantes principales (loadings)

array([[ 0.52237162, -0.26335492,  0.58125401,  0.56561105],
       [ 0.37231836,  0.92555649,  0.02109478,  0.06541577],
       [ 0.72101681, -0.24203288, -0.14089226, -0.6338014 ],
       [-0.26199559,  0.12413481,  0.80115427, -0.52354627]])

In [32]:
X_latent.shape

(150, 4)

In [33]:
X_hat = np.dot(X_latent, pca.components_)

In [34]:
pd.DataFrame(X_hat).head()

,0,1,2,3
0,-0.900681,1.032057,-1.341272,-1.312977
1,-1.143017,-0.124958,-1.341272,-1.312977
2,-1.385353,0.337848,-1.398138,-1.312977
3,-1.506521,0.106445,-1.284407,-1.312977
4,-1.021849,1.263460,-1.341272,-1.312977


In [35]:
pd.DataFrame(X).head()

,0,1,2,3
0,-0.900681,1.032057,-1.341272,-1.312977
1,-1.143017,-0.124958,-1.341272,-1.312977
2,-1.385353,0.337848,-1.398138,-1.312977
3,-1.506521,0.106445,-1.284407,-1.312977
4,-1.021849,1.263460,-1.341272,-1.312977


In [36]:
np.mean((X - X_hat) ** 2)  # sqrt...

np.float64(3.216250556803849e-30)

Comme évoqué en introduction, le but est de réduire la dimensionnalité, donc de ne
conserver que `n_components` composantes principales, avec `n_components`
strictement/très inférieur à la dimensionnalité.

In [37]:
# nombre de composantes pour conserver s % de la variance
s = 95
cumulative_variance = np.cumsum(pca.explained_variance_ratio_) * 100

In [38]:
cumulative_variance

array([ 72.77045209,  95.80097536,  99.48480732, 100.        ])

In [39]:
n_components = np.argmax(cumulative_variance >= s) + 1

In [40]:
n_components

np.int64(2)

In [41]:
# ACP avec ce nombre optimal de composantes
pca_optimal = skl.decomposition.PCA(n_components=n_components)
pca_optimal.fit(X)
X_pca_optimal = pca_optimal.fit_transform(X)

In [42]:
# reconstruction des données dans l'espace réel
X_hat = np.dot(X_pca_optimal, pca_optimal.components_)

In [43]:
np.mean((X - X_hat) ** 2)

np.float64(0.04199024638518027)

In [ ]:
# dé-normalisation pour revenir à l'échelle d'origine ?
# X_reconstructed_real = scaler.inverse_transform(X_hat)

In [ ]:
# Attention au calcul de l'erreur de reconstruction dans ce cas...
# reconstruction_error = np.mean((X - X_reconstructed_original) ** 2)

In [44]:
# Workflow complet
def apply_pca(X, variance_threshold=0.95):
    """Apply standard scaler and PCA to a dataset.

    Parameters
    ----------
    X : array_like
        Input dataset (features: the real variables).
    variance_threshold : float
        Threshold for cumulative explained variance.
        Between 0 and 1.
    
    Returns
    -------
    X_latent : ndarray
        The transformed dataset (the latent variables).
    n_components : int
        Number of components retained.
    """
    # Scaling
    scaler = skl.preprocessing.StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Initial PCA with all components
    pca_full = skl.decomposition.PCA()
    pca_full.fit(X_scaled)
    
    # Number of components determined by variance threshold
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = np.argmax(cumulative_variance >= variance_threshold) + 1
    
    # PCA with optimized number of components
    pca = skl.decomposition.PCA(n_components=n_components)
    X_latent = pca.fit_transform(X_scaled)
    
    return X_latent, n_components

In [45]:
# Exemple d'utilisation de la fonction
X_pca, optimal_components = apply_pca(X, variance_threshold=0.95)
print(f"\nNombre optimal de composantes avec la fonction : {optimal_components}")
print(f"Formes des données originales et transformées : {X.shape} → {X_pca.shape}")


Nombre optimal de composantes avec la fonction : 2
Formes des données originales et transformées : (150, 4) → (150, 2)


#### Exercice

Ajouter au workflow complet le calcul de l'erreur de reconstruction (adimensionnel ou pas, à voir).

## II. Échantillonnage dans l'espace latent

...